# Spark Declarative Pipelines
## Declare your tables. Let Spark build, order, and validate the pipeline.

We'll build a **bronze → silver → gold** medallion over real food-delivery order events and land it as Delta tables in **Unity Catalog** — writing almost entirely **SQL**.

Two things to watch for:
1. **How little code** this takes — the transforms are plain `SELECT`s. No orchestration, no write statements, no manual ordering.
2. **SDP validates the whole dependency graph before it runs a thing** — it catches a typo'd table reference up front, instead of failing halfway through a job.

---
## 1. The pipeline is mostly SQL
Bronze is the only Python — SDP can't read raw files from a SQL `FROM`, so we read the two source files here. Everything downstream is declarative SQL.

In [ ]:
from IPython.display import Code
Code(filename='/home/jovyan/demos/sdp-medallion/transformations/bronze.py')

**Silver** — parse the JSON body, derive time features, join the city dimension. Pure SQL. SDP sees `orders_bronze` and `dim_locations` in the `FROM`/`JOIN` and wires the dependency automatically:

In [ ]:
Code(filename='/home/jovyan/demos/sdp-medallion/transformations/silver_orders_enriched.sql')

**Gold** — a revenue rollup per brand. Again: just a `SELECT`. No write code; the `CREATE MATERIALIZED VIEW` *is* the table.

In [ ]:
Code(filename='/home/jovyan/demos/sdp-medallion/transformations/gold_brand_summary.sql')

> That's the whole transformation layer. You declared **what** each table is; SDP figures out **how** — the DAG (`bronze → silver → gold`), the execution order, and the writes.

---
## 2. The superpower: SDP validates the DAG *before* it runs
With imperative Spark, a typo in a table name blows up at runtime — maybe deep into a long job. SDP builds the dependency graph and checks it with a **dry-run** first. Here's a pipeline with a deliberate bug:

```sql
CREATE MATERIALIZED VIEW brand_revenue AS
SELECT brand_id, sum(amount) AS revenue
FROM orderz          -- typo: should be 'orders'
GROUP BY brand_id;
```

In [ ]:
# Write the broken pipeline to a temp project and dry-run it (no UC, no writes — just graph validation).
import os, subprocess, textwrap
os.makedirs('/tmp/dep_check/transformations', exist_ok=True)
open('/tmp/dep_check/spark-pipeline.yml','w').write(textwrap.dedent('''\
    name: dep_check
    catalog: spark_catalog
    schema: default
    storage: file:///tmp/dep_check_storage
    libraries: [{glob: {include: transformations/**}}]'''))
open('/tmp/dep_check/transformations/orders.sql','w').write(
    "CREATE MATERIALIZED VIEW orders AS SELECT id AS order_id, id % 5 AS brand_id, round(id*9.99,2) AS amount FROM range(1000);")
open('/tmp/dep_check/transformations/brand_revenue.sql','w').write(
    "CREATE MATERIALIZED VIEW brand_revenue AS SELECT brand_id, round(sum(amount),2) AS revenue FROM orderz GROUP BY brand_id;")
out = subprocess.run(['spark-pipelines','dry-run'], cwd='/tmp/dep_check', capture_output=True, text=True)
print('\n'.join(l for l in (out.stdout+out.stderr).splitlines() if 'NOT_FOUND' in l or 'Failed to resolve' in l or 'orderz' in l)[:6] or ['(see full output)'])

**SDP caught it before touching any data** — `TABLE_OR_VIEW_NOT_FOUND: orderz`, with the exact line. Fix the typo and the graph validates clean. That's the value of declaring your dependencies.

---
## 3. Run the real medallion → Unity Catalog
Now the real thing. SDP runs the layers in dependency order and registers catalog-managed Delta tables in your namespace.

In [ ]:
import json, os, re, subprocess, urllib.request
NS = (os.environ.get('DEMO_NS','medallion_demo').rstrip('_')) or 'medallion_demo'
UC = 'https://uc.openlakehousedemos.dev/api/2.1/unity-catalog'
TOK = os.environ.get('UC_TOKEN','')
DEMO = os.environ.get('SDP_DEMO_DIR', '/home/jovyan/demos/sdp-medallion')
H = {'Authorization': f'Bearer {TOK}', 'Content-Type': 'application/json'}

# point the pipeline at THIS presenter's schema (so concurrent runs don't collide)
spec = open(f'{DEMO}/spark-pipeline.yml').read()
open(f'{DEMO}/spark-pipeline.yml','w').write(re.sub(r'(?m)^schema:.*', f'schema: {NS}', spec))

def _uc(url, method, data=None):
    try: urllib.request.urlopen(urllib.request.Request(url, method=method, data=data, headers=H))
    except Exception: pass

_uc(f'{UC}/schemas', 'POST', json.dumps({'name': NS, 'catalog_name': 'managed_demo'}).encode())
# clean slate (UC has no truncate; drop before re-run)
for t in ('gold_brand_summary','gold_hourly_metrics','orders_enriched','orders_bronze','dim_locations'):
    _uc(f'{UC}/tables/managed_demo.{NS}.{t}', 'DELETE')

print(subprocess.run(['spark-pipelines','run'], cwd=DEMO, capture_output=True, text=True).stdout[-1500:])

---
## 4. The result
Query the gold layer straight from Unity Catalog and chart it:

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
bs = spark.table(f'managed_demo.{NS}.gold_brand_summary').orderBy('total_revenue', ascending=False).toPandas()
bs

In [ ]:
import matplotlib.pyplot as plt
top = bs.head(10)[::-1]
plt.figure(figsize=(8,4))
plt.barh(top['brand_name'], top['total_revenue'])
plt.xlabel('revenue'); plt.title(f'Revenue by brand  (managed_demo.{NS})'); plt.tight_layout(); plt.show()

---
### Recap
- The transforms were plain SQL `SELECT`s; SDP inferred the DAG and handled ordering + writes.
- `dry-run` caught a bad dependency before any data moved.
- The gold tables are now in Unity Catalog under `managed_demo.<your-schema>` — open the UC console to browse them.